In [ ]:
import numpy as np
import random
from collections import Counter, defaultdict
import time
import pandas as pd

In [ ]:
# =========================
# 固定参数
# =========================
gamma = 0.9
alpha_A = 0.1
alpha_B = 0.1
epsilon = 0.05

num_episodes = 2000
max_steps_per_episode = 100
num_runs = 100
threshold = 30

states = ["CC", "CD", "DC", "DD"]
num_states = 4
actions = [0, 1]

In [ ]:
def action_to_char(a):
    return "C" if a == 1 else "D"

payoff_matrix = {
    ("CC", "CC"): (4,4), ("CC", "CD"): (1,5), ("CC", "DC"): (5,1), ("CC", "DD"): (2,2),
    ("CD", "CC"): (4,4), ("CD", "CD"): (1,5), ("CD", "DC"): (5,1), ("CD", "DD"): (2,2),
    ("DC", "CC"): (4,4), ("DC", "CD"): (1,5), ("DC", "DC"): (5,1), ("DC", "DD"): (2,2),
    ("DD", "CC"): (4,4), ("DD", "CD"): (1,5), ("DD", "DC"): (5,1), ("DD", "DD"): (2,2)
}

def compute_long_term_value(strategy_A, strategy_B):
    P = np.zeros((4,4))
    R_A = np.zeros(4)
    R_B = np.zeros(4)

    for i, s in enumerate(states):
        aA = strategy_A[i]
        aB = strategy_B[i]
        next_state = ("C" if aA else "D") + ("C" if aB else "D")
        j = states.index(next_state)

        P[i, j] = 1
        R_A[i], R_B[i] = payoff_matrix[(s, next_state)]

    I = np.eye(4)
    V_A = np.linalg.inv(I - gamma * P).dot(R_A)
    V_B = np.linalg.inv(I - gamma * P).dot(R_B)
    return V_A, V_B

def compute_non_strategy_vector(Q):
    vec = []
    for row in Q:
        max_idx = np.argmax(row)
        non_max = np.delete(row, max_idx)[0]
        vec.append(non_max)
    return np.array(vec)

In [ ]:
# =========================
# 存储所有实验的数据
# =========================
all_stats = []          # 用于 Manifold Stats
joint_stats_list = []   # 用于 Joint Distribution

In [ ]:
# =========================
# 外层循环：25次实验
# =========================
for exp_id in range(25):
    print(f"\n{'='*60}")
    print(f"Experiment {exp_id+1} / 25")
    print('='*60)

    # 为每次实验生成一个独立的基础种子
    base_seed = int(time.time() * 1000000) % (2**32) + exp_id * 1234567
    print(f"Base seed for this experiment: {base_seed}")

    # 重置统计容器
    strategy_counter = Counter()
    manifold_stats = defaultdict(lambda: {
        "count": 0,
        "optimistic": 0,
        "pessimistic": 0,
        "invariant": 0,
        "non_invariant": 0
    })
    joint_stats = {
        ("optimistic", "invariant"): 0,
        ("optimistic", "non_invariant"): 0,
        ("pessimistic", "invariant"): 0,
        ("pessimistic", "non_invariant"): 0
    }

    # =========================
    # 内层循环：100个run
    # =========================
    for run_id in range(num_runs):
        seed = base_seed + run_id
        random.seed(seed)
        np.random.seed(seed)

        Q_A = np.zeros((4,2))
        Q_B = np.zeros((4,2))

        # Q-learning 训练
        for episode in range(num_episodes):
            for initial_state in range(4):
                state = initial_state
                for step in range(max_steps_per_episode):
                    # A 选择动作
                    if random.random() < epsilon:
                        action_A = random.choice(actions)
                    else:
                        action_A = np.random.choice(np.where(Q_A[state] == np.max(Q_A[state]))[0])

                    # B 选择动作
                    if random.random() < epsilon:
                        action_B = random.choice(actions)
                    else:
                        action_B = np.random.choice(np.where(Q_B[state] == np.max(Q_B[state]))[0])

                    joint = action_to_char(action_A) + action_to_char(action_B)
                    reward_A, reward_B = payoff_matrix[(states[state], joint)]
                    next_state = states.index(joint)

                    # 更新 Q 表
                    Q_A[state, action_A] += alpha_A * (
                        reward_A + gamma * np.max(Q_A[next_state]) - Q_A[state, action_A]
                    )
                    Q_B[state, action_B] += alpha_B * (
                        reward_B + gamma * np.max(Q_B[next_state]) - Q_B[state, action_B]
                    )

                    state = next_state

        # 提取最终策略
        strategy_A = ""
        strategy_B = ""
        for s in range(4):
            a = np.random.choice(np.where(Q_A[s] == np.max(Q_A[s]))[0])
            b = np.random.choice(np.where(Q_B[s] == np.max(Q_B[s]))[0])
            strategy_A += action_to_char(a)
            strategy_B += action_to_char(b)

        strategy_counter[(strategy_A, strategy_B)] += 1
        key = (strategy_A, strategy_B)

        # 计算非策略值
        non_A = compute_non_strategy_vector(Q_A)
        non_B = compute_non_strategy_vector(Q_B)

        # 乐观/悲观判定
        non_avg = np.array([np.mean(non_A), np.mean(non_B)])
        pessimistic = np.any(non_avg < threshold)
        optimistic = np.all(non_avg >= threshold)

        # 强不变流形判定
        stratA_bin = np.array([1 if c=="C" else 0 for c in strategy_A])
        stratB_bin = np.array([1 if c=="C" else 0 for c in strategy_B])
        V_A, V_B = compute_long_term_value(stratA_bin, stratB_bin)

        invariant = True
        for s in range(4):
            if not (non_A[s] < V_A[s] and non_B[s] < V_B[s]):
                invariant = False
                break
        non_invariant = not invariant

        # 统计
        manifold_stats[key]["count"] += 1
        if optimistic:
            manifold_stats[key]["optimistic"] += 1
        if pessimistic:
            manifold_stats[key]["pessimistic"] += 1
        if invariant:
            manifold_stats[key]["invariant"] += 1
        else:
            manifold_stats[key]["non_invariant"] += 1

        if optimistic and invariant:
            joint_stats[("optimistic", "invariant")] += 1
        elif optimistic and non_invariant:
            joint_stats[("optimistic", "non_invariant")] += 1
        elif pessimistic and invariant:
            joint_stats[("pessimistic", "invariant")] += 1
        elif pessimistic and non_invariant:
            joint_stats[("pessimistic", "non_invariant")] += 1

    # =========================
    # 输出本次实验的结果
    # =========================
    print("\n=== Strategy Frequency ===")
    for k, v in strategy_counter.most_common():
        print(k, v)

    print("\n=== Manifold Stats ===")
    for k, v in sorted(manifold_stats.items(), key=lambda x: x[1]["count"], reverse=True):
        print(k, v)

    print("\n=== Joint Distribution ===")
    total = sum(joint_stats.values())
    for k, v in joint_stats.items():
        print(f"{k}: {v}  (prob = {v/total:.4f})")

    # =========================
    # 将本次实验的 Manifold Stats 添加到全局列表
    # =========================
    for strategy, stats in manifold_stats.items():
        all_stats.append({
            "实验批次": exp_id + 1,
            "策略流形": str(strategy),
            "总数": stats["count"],
            "乐观": stats["optimistic"],
            "悲观": stats["pessimistic"],
            "不变流形": stats["invariant"],
            "非不变流形": stats["non_invariant"]
        })

    # =========================
    # 将本次实验的 Joint Distribution 添加到全局列表
    # =========================
    joint_stats_list.append({
        "实验批次": exp_id + 1,
        "乐观-不变流形": joint_stats[("optimistic", "invariant")],
        "悲观-不变流形": joint_stats[("pessimistic", "invariant")],
        "乐观-非不变流形": joint_stats[("optimistic", "non_invariant")],
        "悲观-非不变流形": joint_stats[("pessimistic", "non_invariant")]
    })

    print("\n" + "="*60 + "\n")

In [ ]:
# =========================
# 所有实验结束后，保存到 Excel 的两个 Sheet
# =========================
df_manifold = pd.DataFrame(all_stats)
df_manifold = df_manifold.sort_values(["实验批次", "总数"], ascending=[True, False])

df_joint = pd.DataFrame(joint_stats_list)
df_joint = df_joint.sort_values("实验批次")

with pd.ExcelWriter("manifold_stats_all_experiments.xlsx") as writer:
    df_manifold.to_excel(writer, sheet_name="Manifold Stats", index=False)
    df_joint.to_excel(writer, sheet_name="Joint Distribution", index=False)

print("数据已保存到 manifold_stats_all_experiments.xlsx")